In [1]:
from dotenv import load_dotenv
from typing import Any
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5-mini", temperature=0.3)

In [4]:
docs = [
    Document(
        page_content=(
            "Biotechnology companies are developing novel protein-based therapies that target specific "
            "disease pathways with unprecedented precision. Synthetic biology techniques allow scientists "
            "to engineer microorganisms that produce pharmaceutical compounds at industrial scale. "
            "Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, "
            "making treatments for autoimmune diseases and cancers more accessible. Microbiome research is "
            "revealing how manipulating gut bacteria can influence everything from mental health to "
            "metabolic disorders."
        ),
        metadata={"topic": "biotechnology"},
    ),
    Document(
        page_content=(
            "Zero-trust architecture has become the gold standard for enterprise network security, "
            "requiring continuous verification rather than relying on perimeter defenses. Machine learning "
            "models now detect anomalous network behavior in real time, reducing the window between "
            "intrusion and detection from months to minutes. Ransomware attacks on critical infrastructure "
            "have forced governments to establish mandatory incident reporting requirements for healthcare "
            "and energy sectors. Post-quantum cryptography standards are being finalized to protect "
            "sensitive data against future quantum computing threats."
        ),
        metadata={"topic": "cybersecurity"},
    ),
    Document(
        page_content=(
            "Brain-computer interfaces are enabling paralyzed patients to control prosthetic limbs and "
            "communicate using only their neural signals. Optogenetics allows researchers to activate or "
            "silence specific neuron populations with light, accelerating the understanding of neural "
            "circuit function and disease. Advanced neuroimaging techniques using fMRI and "
            "magnetoencephalography are mapping brain connectivity with millimeter precision, unlocking "
            "new treatments for depression and PTSD. Neurofeedback therapies are showing promise for "
            "cognitive rehabilitation following traumatic brain injuries."
        ),
        metadata={"topic": "neuroscience"},
    ),
    Document(
        page_content=(
            "Perovskite solar cells have achieved efficiency ratings exceeding 33%, surpassing traditional "
            "silicon panels and promising dramatically lower manufacturing costs. Grid-scale battery "
            "storage using iron-air and sodium-ion technologies is making renewable energy dispatchable "
            "around the clock without relying on rare earth metals. Offshore floating wind farms are "
            "expanding into deep-water regions previously inaccessible to fixed-foundation turbines, "
            "multiplying available wind energy capacity. Green hydrogen produced via electrolysis is "
            "emerging as a critical energy carrier for decarbonizing heavy industry and long-haul "
            "transport."
        ),
        metadata={"topic": "renewable_energy"},
    ),
    Document(
        page_content=(
            "Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive "
            "procedures with sub-millimeter precision, reducing patient recovery times significantly. "
            "Collaborative robots in manufacturing now work safely alongside humans using advanced "
            "computer vision and force sensing, without the need for physical barriers. Autonomous mobile "
            "robots are transforming warehouse logistics, optimizing pick-and-place operations and "
            "reducing fulfillment errors. Soft robots inspired by biological organisms are being developed "
            "for delicate tasks in agriculture, search-and-rescue, and medical drug delivery."
        ),
        metadata={"topic": "robotics"},
    ),
    Document(
        page_content=(
            "Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, "
            "enabling single-letter corrections to the genome without creating double-strand breaks. "
            "Gene therapy trials using adeno-associated virus vectors have achieved functional cures for "
            "hemophilia B and spinal muscular atrophy. Epigenome editing tools allow researchers to "
            "switch genes on or off without altering the underlying DNA sequence, opening new avenues "
            "for treating complex diseases. Polygenic risk scoring combined with germline analysis is "
            "enabling predictive medicine that identifies disease susceptibility decades before symptoms "
            "appear."
        ),
        metadata={"topic": "genetic_engineering"},
    ),
]

print(f"Created {len(docs)} documents")

Created 6 documents


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

docs = splitter.split_documents(docs)

In [6]:
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
# QueriesSchema is the structured output the LLM produces — a list of alternative questions
class QueriesSchema(BaseModel):
    queries: list[str] = Field(description="List of 3 alternative versions of the question")


prompt = ChatPromptTemplate.from_template(
    "You are an AI language model assistant. Your task is to generate 3 different versions of "
    "the given user question to retrieve relevant documents from a vector database. "
    "By generating multiple perspectives on the user question, your goal is to help the user "
    "overcome some of the limitations of distance-based similarity search. "
    "Provide these alternative questions separated by newlines.\n\n"
    "Original question: {question}"
)

llm_structured_output = llm.with_structured_output(QueriesSchema)

# with_structured_output binds QueriesSchema so the chain always returns a QueriesSchema instance
query_chain = prompt | llm_structured_output


class CustomMultiQueryRetriever(BaseRetriever):
    """Retriever that generates multiple query perspectives via an LLM and deduplicates results."""

    base_retriever: BaseRetriever
    query_chain: Any  # prompt | llm.with_structured_output(QueriesSchema)

    def _generate_queries(self, query: str) -> list[str]:
        result: QueriesSchema = self.query_chain.invoke({"question": query})
        return result.queries

    def _unique_documents(self, documents: list[Document]) -> list[Document]:
        # Deduplicate by page_content, preserving first-occurrence order
        seen: set[str] = set()
        unique: list[Document] = []
        for doc in documents:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique.append(doc)
        return unique

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # Step 1: generate alternative query phrasings
        queries = self._generate_queries(query)
        # Step 2: retrieve docs for each alternative query
        all_docs: list[Document] = []
        for q in queries:
            all_docs.extend(self.base_retriever.invoke(q))
        # Step 3: deduplicate and return
        return self._unique_documents(all_docs)


In [8]:
retriever = CustomMultiQueryRetriever(base_retriever=base_retriever, query_chain=query_chain)

query = "How are modern technologies improving human health?"

# Peek at the alternative queries before seeing results
parsed = query_chain.invoke({"question": query})

print("Generated alternative queries:")
for q in parsed.queries:
    print(f"  - {q}")
print()

results = retriever.invoke(query)
print(f"Retrieved {len(results)} unique documents:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()

Generated alternative queries:
  - Which modern technologies (e.g., AI, wearables, genomics, telemedicine, medical devices) are improving human health, and through what mechanisms?
  - How do innovations like AI-driven diagnostics, remote monitoring, personalized medicine, and robotic surgery impact disease prevention, diagnostic accuracy, treatment effectiveness, and healthcare access?
  - What evidence and measurable outcomes demonstrate that recent technologies have improved population health and individual patient outcomes over the past decade?

Retrieved 6 unique documents:

--- Result 1 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have

--- Result 2 [genetic_engineering] ---
functional cures for hemophilia B and spin